# Chapter 13: Deploying scikit-learn Models in Production

## Ringkasan Bab & Penjelasan Teoretis
Membawa model machine learning dari lingkungan eksperimen ke lingkungan produksi (*production deployment*) membutuhkan arsitektur yang terukur, dapat direproduksi, dan terpantau secara berkelanjutan (*MLOps*).

### 1. Serialisasi & Persistensi Model (`joblib` dan `pickle`)
Agar model tidak perlu dilatih ulang setiap kali aplikasi menerima permintaan prediksi baru, seluruh objek `Pipeline` (mencakup pra-pemrosesan dan estimator terlatih) disimpan ke dalam media penyimpanan menggunakan serialisasi biner:
* **`joblib`**: Sangat direkomendasikan untuk objek `scikit-learn` karena dioptimalkan secara khusus untuk menyimpan *array* NumPy berukuran besar secara efisien.
* **`pickle`**: Modul serialisasi objek standar bawaan Python.

### 2. Pemantauan Akurasi Model (*Model Drift Monitoring*) & Validasi Otomatis
Distribusi data di dunia nyata senantiasa berubah seiring waktu (*data drift* / *concept drift*). Oleh karena itu, sistem produksi wajib:
1. Memantau metrik performa model secara berkala pada *batch* data terbaru.
2. Menerapkan pemeriksaan ambang batas validasi otomatis (*automated deployment gate*) yang memicu peringatan atau pelatihan ulang (*retraining*) apabila akurasi turun di bawah batas minimum yang ditetapkan.

In [ ]:
import joblib
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

# 1. Membangun dan Menyimpan Deployment Pipeline (Serialization dengan joblib & pickle)
wine = load_wine()
X_train, X_test, y_train, y_test = train_test_split(wine.data, wine.target, test_size=0.25, random_state=2024)

production_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('classifier', RandomForestClassifier(n_estimators=100, random_state=2024))
])

production_pipeline.fit(X_train, y_train)

# Simpan dan muat ulang dengan joblib
joblib_filename = "wine_production_pipeline.joblib"
joblib.dump(production_pipeline, joblib_filename)
loaded_joblib_pipeline = joblib.load(joblib_filename)

# Simpan dan muat ulang dengan pickle
pickle_filename = "wine_production_pipeline.pkl"
with open(pickle_filename, "wb") as f:
    pickle.dump(production_pipeline, f)
with open(pickle_filename, "rb") as f:
    loaded_pickle_pipeline = pickle.load(f)

preds = loaded_joblib_pipeline.predict(X_test)
print(f"Loaded Production Pipeline Accuracy on Test Set: {accuracy_score(y_test, preds):.4f}")

# 2. Simulasi Monitoring Akurasi Model Seiring Waktu (Detecting Data Drift) & Automated Validation Gate
np.random.seed(2024)
drift_levels = [0.0, 0.2, 0.5, 1.0, 1.8, 2.5]
accuracy_over_time = []
validation_threshold = 0.85

print("\n--- Automated Deployment & Drift Monitoring Log ---")
for batch_idx, noise_scale in enumerate(drift_levels, start=1):
    # Simulasi pergeseran distribusi data (data drift) pada setiap batch waktu
    X_batch_drifted = X_test + np.random.normal(0, noise_scale, X_test.shape)
    batch_acc = accuracy_score(y_test, loaded_joblib_pipeline.predict(X_batch_drifted))
    accuracy_over_time.append(batch_acc)
    status = "PASSED (Healthy)" if batch_acc >= validation_threshold else "ALERT: Retraining Required!"
    print(f"Batch {batch_idx} (Drift={noise_scale:.1f}) -> Accuracy: {batch_acc:.3f} | Status: {status}")

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(drift_levels) + 1), accuracy_over_time, marker='o', linewidth=2, label='Model Accuracy')
plt.axhline(y=validation_threshold, color='crimson', linestyle='--', label=f'Validation Threshold ({validation_threshold})')
plt.xlabel('Production Time Batch (Increasing Data Drift)')
plt.ylabel('Accuracy')
plt.title('Monitoring Deployed Model Accuracy Over Time')
plt.ylim(0.4, 1.05)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()